In [1]:
source_workspace = "a7ad27b1-6ad7-440f-bf78-030662ef90fd"
source_lakehouse = "6c777342-6ab0-4f34-8ec8-4eb29b1a8ea7"
source_folder = "Files/transactional_data"


target_workspace = "a7ad27b1-6ad7-440f-bf78-030662ef90fd"
target_lakehouse = "d66a8795-61e5-4bd3-97ab-b9eb6508bf5a"
target_schema = "base_transactional"
target_folder = "Files/base/transactional"

table_config = [
    {
        "source_file" : "EFI.csv",
        "target_table" : "efi"
    },
    {
        "source_file" : "TME.csv",
        "target_table" : "tme"
    },
    {
        "source_file" : "ChronoMetrics.csv",
        "target_table" : "chrono_metrics"
    }
]

StatementMeta(, 2ee6f21b-80e6-463a-b53c-6a1893aff68e, 3, Finished, Available, Finished, False)

In [2]:
import re
import pyspark.sql.functions as F

StatementMeta(, 2ee6f21b-80e6-463a-b53c-6a1893aff68e, 4, Finished, Available, Finished, False)

In [3]:
def normalize_column_name(name: str) -> str:
    name = name.strip()
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_").lower()

StatementMeta(, 2ee6f21b-80e6-463a-b53c-6a1893aff68e, 5, Finished, Available, Finished, False)

In [4]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")

StatementMeta(, 2ee6f21b-80e6-463a-b53c-6a1893aff68e, 6, Finished, Available, Finished, False)

DataFrame[]

In [5]:
for cfg in table_config:
    source_file = cfg["source_file"]
    target_table = cfg["target_table"]
    full_table = f"{target_schema}.{target_table}"
    target_path = f"{target_folder}/{target_table}"
    source_file_path = f"abfss://{source_workspace}@onelake.dfs.fabric.microsoft.com/{source_lakehouse}/{source_folder}/{source_file}"

    # Read CSV
    df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("delimiter", ",")
        .load(source_file_path)
    )

    # Normalize column names
    df = df.toDF(*[normalize_column_name(c) for c in df.columns])

    # Deduplicate rows
    source_columns = df.columns
    dup_count = df.count()

    df = df.dropDuplicates(source_columns)
    dedup_count = df.count()
    print(f"{source_file} removed {dup_count - dedup_count} duplicate rows")
    
    # Add metadata
    df = (
        df
        .withColumn("_ingest_date", F.current_timestamp())
        .withColumn("_source_file", F.lit(source_file))
        .withColumn("_source_system", F.lit(source_file_path))
    )

    # Create delta tables
    if not spark.catalog.tableExists(full_table):
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("path", target_path)
            .option("mergeSchema", "true")
            .saveAsTable(full_table)
        )
    else:
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("mergeSchema", "true")
            .save(target_path)
        )

    print(f"Loaded {source_file} into {full_table}")

StatementMeta(, 2ee6f21b-80e6-463a-b53c-6a1893aff68e, 7, Finished, Available, Finished, False)

EFI.csv removed 12 duplicate rows
Loaded EFI.csv into base_transactional.efi
TME.csv removed 0 duplicate rows
Loaded TME.csv into base_transactional.tme
ChronoMetrics.csv removed 0 duplicate rows
Loaded ChronoMetrics.csv into base_transactional.chrono_metrics
